# Transformers

## Begin with one simple question

How can a computer read a sentence and use the important parts to make a useful prediction? A Transformer is one kind of neural network that turns a sequence of pieces (such as words or parts of words) into numbers, lets those pieces exchange information, and then uses the updated numbers for a task.

We will build that idea one small step at a time: split text into tokens, turn tokens into vectors, add their order, let tokens share information with attention, repeat a simple block, and make an output. No earlier knowledge of attention or Transformers is needed.

## Step 1: turn a sentence into a row of numbers

A **sequence** is an ordered list. In text, a tokenizer breaks a sentence into **tokens**. A token might be a whole short word, a piece of a longer word, punctuation, or a special marker. The tokenizer assigns each token an integer called a **token ID**. The ID is just a lookup address; the number itself does not mean the word is larger, better, or more important.

A learned table turns each ID into an **embedding**: a short list, or vector, of numbers. For a tiny made-up example, `cat` might map to `[0.2, -0.1, 0.7]`. A real model uses many more numbers, and training adjusts them. Similar uses may lead to useful patterns in these vectors, but no single coordinate has a guaranteed human-readable meaning.

The same token can appear in different places, so the model also needs **position information**. We add a position vector to each token embedding (or use another positional method). Now the representation says both what token is present and where it occurs.

## Step 2: see the whole journey before opening a block

```text
sentence
   |
   v
split into tokens -> token IDs -> embeddings + position information
                                      |
                                      v
                         Transformer block (repeat)
                                      |
                                      v
                       task-specific output layer
                                      |
                                      v
                              prediction
```

A **block** is a small group of calculations. A Transformer usually stacks several blocks, meaning the output of one becomes the input to the next. In each block, attention lets token positions share information; a small feed-forward network then adjusts each position's representation. Skip connections and normalization help keep these repeated calculations trainable.

## Step 3: let tokens share useful information

Imagine `The puppy carried the ball`. While building the representation for `carried`, the model can use information from `puppy` and `ball`, not only the previous word. This information-sharing step is called **self-attention**: each token looks at other allowed tokens in the same sequence and gathers a weighted mixture of their information.

Each token makes three learned descriptions. A **query** is what this token is looking for; a **key** is what another token offers for matching; a **value** is the information that can be passed along. The model compares a query with each key to make a match score. A function called **softmax** turns those scores into weights that add to one, and the weights determine how much of each value to mix. For scores $[0,1]$, softmax gives about $[0.27,0.73]$. If the corresponding values are $2$ and $6$, the result is $0.27(2)+0.73(6)=4.92$: more information came from the second value because it had the larger score. Real models use learned vector descriptions rather than these tiny numbers.

The word **self** means the queries, keys, and values come from this same sequence. In cross-attention, queries come from one sequence while keys and values come from another.

## Step 4: refine each token, then keep a path for the old information

After attention, each token has gathered information from other positions. Most Transformers use **multi-head attention**: several smaller attention reads happen in parallel, and their results are combined. This gives the model more than one learned way to compare positions.

A **feed-forward network** (FFN) is a small neural network applied to each position separately. It changes that position's representation, but it does not mix information between different positions; attention did that.

A **residual connection**, also called a skip connection, adds the incoming representation back to the result of a step. This gives the model a direct path for keeping useful old information while learning an update. **Normalization** gently rescales the numbers in a representation so repeated layers are easier to train. One common pre-normalized block is:

$$u=x+\operatorname{Attention}(\operatorname{Norm}(x)),\qquad y=u+\operatorname{FFN}(\operatorname{Norm}(u)).$$

Here $x$ is the incoming vector for one position, $u$ is the vector after attention and its skip connection, and $y$ is the block's output for that position. The same attention step also shares information across positions. For a one-number illustration that ignores normalization, if the old value is $2$ and attention adds $0.5$, the skip connection makes $u=2.5$. If the FFN then adds $-0.25$, the second skip connection gives $y=2.25$. Real vectors have many coordinates. Some models arrange normalization differently, but the roles are similar.

## Step 5: turn the final numbers into an answer

After several blocks, the model has a new vector for every token. A small task layer reads these vectors and produces the answer. For **classification**, it might turn a sentence representation into scores for labels such as positive or negative. These raw scores are called **logits**; softmax can turn them into values that add to one, which are often used as class probabilities.

For **next-token prediction**, the output layer instead makes one score for every token in the vocabulary at each position. A softmax turns those scores into a next-token distribution. During training, the correct next token is known, so predictions for many positions can be calculated in parallel. During generation, the next token is not known yet: choose or sample one, add it to the input, and run the model again.

A **mask** is a rule that hides positions the model must not use. An encoder for understanding can usually read tokens on both sides. A decoder for next-token generation uses a **causal mask**, which lets a position read itself and earlier tokens but blocks later tokens. An encoder-decoder model combines an encoder's source representations with a masked decoder that generates the target.

## Step 6: understand the encoder and decoder chain

A Transformer is built by chaining blocks: the output of block 1 becomes the input to block 2, and so on. Each block updates every token's vector, while attention lets tokens exchange information. The chain turns simple input vectors into richer, context-aware vectors.

There are three common arrangements:

| Arrangement | Information flow | Often used for |
| --- | --- | --- |
| Encoder-only | Each input token can read the other input tokens in both directions | Classification, search, and labeling | 
| Decoder-only | Each token can read only itself and earlier tokens | Predicting and generating the next token | 
| Encoder-decoder | Encoder reads the source; decoder reads its own prefix and attends to the source | Translation and other input-to-output tasks |

In an encoder-decoder model, the decoder's **cross-attention** is the link back to the encoder: decoder queries read from the encoder's output vectors. These are different ways to arrange Transformer blocks, not different meanings of the word “Transformer.”

## Step 7: know what Transformers are good at, and what they cost

A recurrent neural network (RNN) reads a sequence one position after another, carrying a running state forward. A Transformer uses attention to let positions exchange information directly. This makes it possible to calculate many positions in parallel during training, rather than waiting for the previous position's new state. For generated text, however, each new token still depends on the tokens already generated, so generation proceeds step by step.

In an encoder with unrestricted dense attention, each position compares itself with every other position. If a sequence grows from 10 positions to 20, the number of comparisons grows from 100 to 400. A causal decoder hides future positions, so it does fewer comparisons, but its cost still grows roughly with the square of sequence length. Flexible access to context is useful, but long sequences need more memory and computation. Some newer attention methods reduce this cost, but they are beyond this introduction.

Transformers are useful when relationships between distant positions matter and there is enough data and compute to train the model. They are not automatically better than recurrent networks or simpler methods. They need position information, correct masks, suitable training examples, and careful evaluation.

**Recap:** split the input into tokens; map tokens to vectors and add order; use attention to share information between positions; use a feed-forward network to refine each position; repeat the block; then use an output layer for the task. That is the basic Transformer.